In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [48]:
# Kaggle dependency setup (force official MediaPipe package)
!pip -q uninstall -y mediapipe mediapipe-silicon mediapipe-nightly
!pip -q install --no-cache-dir "protobuf<5" "mediapipe==0.10.14" "opencv-python-headless<4.11" scikit-learn==1.5.1

print("✅ Dependencies installed. If this is a fresh install, restart kernel before running next cells.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 118.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 51.7 MB/s eta 0:00:00 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 250.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 166.2 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
category-encoders 2.9.0 requires scikit-learn>=1.6.0, but you have scikit-learn 1.5.1 which is incompatible.
a2a-sdk 0.3.23 requires protobuf>=5.29.5, but you have protobuf 4.25.8 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible

In [74]:
import os
import warnings

# Reduce verbose native logs from TensorFlow/MediaPipe
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import cv2
import json
import random
import numpy as np
import pandas as pd
from glob import glob

import mediapipe as mp
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Hide known noisy protobuf deprecation warning in Kaggle runtime
warnings.filterwarnings(
    "ignore",
    message=r"SymbolDatabase.GetPrototype\(\) is deprecated.*",
    category=UserWarning,
    module="google.protobuf.symbol_database",
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


TensorFlow: 2.19.0


In [21]:
import requests, zipfile, io

url = "https://data.mendeley.com/public-api/zip/8fmvr9m98w/download/3"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall("/kaggle/working/signalphaset")

In [ ]:
!unzip -q /kaggle/working/signalphaset/SignAlphaSet/ASL_dynamic.zip

In [40]:
# =========================
# Configuration
# =========================
DATA_ROOT = "/kaggle/working/ASL_dynamic"  # Point this to your dataset root if needed
VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv", ".webm")
FRAME_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Your exact school-project vocabulary (5 words)
TARGET_WORDS = ["hello", "thankyou", "yes", "no", "sorry"]
REQUIRE_ALL_TARGET_WORDS = True

# Choose source type: "video", "frames", or "auto"
# - video: use clips only
# - frames: use frame folders only
# - auto: prefer videos if found, otherwise frames
PREFERRED_SOURCE = "video"

MAX_FRAMES = 40  # Number of timesteps per sample
MAX_SAMPLES_PER_CLASS = 250  # Safety cap for speed/memory
IMG_SIZE = (320, 240)
TEST_SIZE = 0.15
VAL_SIZE = 0.15
BATCH_SIZE = 8   # small dataset -> smaller batch gives more steps/epoch
EPOCHS = 45

CACHE_DIR = "/kaggle/working/cache_landmarks"
os.makedirs(CACHE_DIR, exist_ok=True)

print("Config ready.")

import re


def normalize_token(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]", "", s)
    return s


TARGET_MAP = {normalize_token(w): w for w in TARGET_WORDS}


def infer_label_from_path(path: str):
    parts = os.path.normpath(path).split(os.sep)
    for p in reversed(parts):
        t = normalize_token(p)
        if t in TARGET_MAP:
            return TARGET_MAP[t]
    return None


def discover_video_files(data_root):
    files = []
    for ext in VIDEO_EXTS:
        files.extend(glob(os.path.join(data_root, "**", f"*{ext}"), recursive=True))
    files = sorted(list(set(files)))

    records = []
    for fp in files:
        label = infer_label_from_path(fp)
        if label is None:
            continue
        records.append((fp, label, "video"))

    return pd.DataFrame(records, columns=["media_path", "label", "source_type"])


def discover_frame_dirs(data_root):
    frame_dirs = []
    for root, _, files in os.walk(data_root):
        img_files = [f for f in files if f.lower().endswith(FRAME_EXTS)]
        if len(img_files) > 0:
            frame_dirs.append(root)

    frame_dirs = sorted(list(set(frame_dirs)))
    records = []
    for d in frame_dirs:
        label = infer_label_from_path(d)
        if label is None:
            continue
        records.append((d, label, "frames"))

    return pd.DataFrame(records, columns=["media_path", "label", "source_type"])


df_video = discover_video_files(DATA_ROOT)
df_frames = discover_frame_dirs(DATA_ROOT)

print("Video samples found:", len(df_video))
print("Frame folders found:", len(df_frames))

if PREFERRED_SOURCE == "video":
    df_all = df_video if len(df_video) > 0 else df_frames
elif PREFERRED_SOURCE == "frames":
    df_all = df_frames if len(df_frames) > 0 else df_video
else:
    df_all = df_video if len(df_video) > 0 else df_frames

if len(df_all) == 0:
    raise ValueError(
        "No usable samples found. Ensure your path has videos or frame folders for target words."
    )

df = df_all[df_all["label"].isin(TARGET_WORDS)].copy()

if len(df) == 0:
    raise ValueError(
        f"Found data, but none matched TARGET_WORDS={TARGET_WORDS}. Check folder names."
    )

found_labels = sorted(df["label"].unique().tolist())
missing_labels = sorted(set(TARGET_WORDS) - set(found_labels))

if REQUIRE_ALL_TARGET_WORDS and missing_labels:
    raise ValueError(f"Missing target classes in data: {missing_labels}")

# Optional cap per class for speed
df = (
    df.groupby("label", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), MAX_SAMPLES_PER_CLASS), random_state=SEED))
    .reset_index(drop=True)
)

print("Using source type:", df["source_type"].iloc[0])
print("Selected labels:", sorted(df["label"].unique()))
print("Samples after cap:", len(df))
print(df["label"].value_counts())

Config ready.
Video samples found: 40
Frame folders found: 40
Using source type: video
Selected labels: ['hello', 'no', 'sorry', 'thankyou', 'yes']
Samples after cap: 40
label
hello       10
sorry       10
thankyou    10
no           5
yes          5
Name: count, dtype: int64


/tmp/ipykernel_55/1335428014.py:120: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), MAX_SAMPLES_PER_CLASS), random_state=SEED))


In [41]:
# Train/Val/Test split with robust stratification for small datasets
label_counts = df["label"].value_counts()
can_stratify = label_counts.min() >= 3

if can_stratify:
    train_df, test_df = train_test_split(
        df, test_size=TEST_SIZE, random_state=SEED, stratify=df["label"]
    )

    train_counts = train_df["label"].value_counts()
    can_stratify_val = train_counts.min() >= 2
    train_df, val_df = train_test_split(
        train_df,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=train_df["label"] if can_stratify_val else None,
    )
else:
    print("[WARN] Very small class counts detected. Using non-stratified split.")
    train_df, test_df = train_test_split(
        df, test_size=TEST_SIZE, random_state=SEED, stratify=None
    )
    train_df, val_df = train_test_split(
        train_df, test_size=VAL_SIZE, random_state=SEED, stratify=None
    )

le = LabelEncoder()
le.fit(train_df["label"])

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(split_name, len(split_df))
    print(split_df["label"].value_counts())

class_names = list(le.classes_)
num_classes = len(class_names)
print("Classes:", class_names)
print("num_classes:", num_classes)

train 28
label
sorry       8
hello       7
thankyou    7
no          3
yes         3
Name: count, dtype: int64
val 6
label
hello       2
no          1
yes         1
thankyou    1
sorry       1
Name: count, dtype: int64
test 6
label
thankyou    2
hello       1
sorry       1
yes         1
no          1
Name: count, dtype: int64
Classes: ['hello', 'no', 'sorry', 'thankyou', 'yes']
num_classes: 5


In [75]:
# =========================
# MediaPipe Landmark Extraction
# Output shape per sample: [MAX_FRAMES, 126]
# 126 = 2 hands * 21 landmarks * 3 coords
# =========================
mp_hands = mp.python.solutions.hands
print("Using MediaPipe Hands from mp.solutions.hands")

def _empty_hand():
    return np.zeros((21, 3), dtype=np.float32)

def extract_hand_vector_from_frame(frame_bgr, hands_model):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    result = hands_model.process(frame_rgb)

    left = _empty_hand()
    right = _empty_hand()

    if result.multi_hand_landmarks and result.multi_handedness:
        for hand_lm, handedness in zip(
            result.multi_hand_landmarks, result.multi_handedness
        ):
            pts = np.array(
                [[lm.x, lm.y, lm.z] for lm in hand_lm.landmark], dtype=np.float32
            )
            label = handedness.classification[0].label.lower()  # 'left' or 'right'
            if label == "left":
                left = pts
            else:
                right = pts

    feat = np.concatenate([left.reshape(-1), right.reshape(-1)], axis=0)
    return feat

def uniform_indices(n, max_len):
    if n <= max_len:
        return np.arange(n)
    return np.linspace(0, n - 1, max_len).astype(int)

def sort_key_natural(name):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", name)]

def pad_or_trim(seq, max_frames=MAX_FRAMES):
    seq = np.array(seq, dtype=np.float32)
    if len(seq) < max_frames:
        pad = np.zeros((max_frames - len(seq), 126), dtype=np.float32)
        seq = np.vstack([seq, pad])
    elif len(seq) > max_frames:
        seq = seq[:max_frames]
    return seq

def extract_sequence_from_video(video_path, hands_model, max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total <= 0:
        frames = []
        while True:
            ok, fr = cap.read()
            if not ok:
                break
            frames.append(fr)
        cap.release()

        if len(frames) == 0:
            return np.zeros((max_frames, 126), dtype=np.float32)

        idx = uniform_indices(len(frames), max_frames)
        seq = []
        for i in idx:
            fr = cv2.resize(frames[i], IMG_SIZE)
            seq.append(extract_hand_vector_from_frame(fr, hands_model))
        return pad_or_trim(seq, max_frames)

    idx_set = set(uniform_indices(total, max_frames).tolist())
    seq = []
    frame_idx = 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        if frame_idx in idx_set:
            fr = cv2.resize(fr, IMG_SIZE)
            seq.append(extract_hand_vector_from_frame(fr, hands_model))
        frame_idx += 1
    cap.release()
    return pad_or_trim(seq, max_frames)

def extract_sequence_from_frames_dir(frames_dir, hands_model, max_frames=MAX_FRAMES):
    frame_files = [
        os.path.join(frames_dir, f)
        for f in os.listdir(frames_dir)
        if f.lower().endswith(FRAME_EXTS)
    ]
    frame_files = sorted(
        frame_files, key=lambda x: sort_key_natural(os.path.basename(x))
    )

    if len(frame_files) == 0:
        return np.zeros((max_frames, 126), dtype=np.float32)

    idx = uniform_indices(len(frame_files), max_frames)
    seq = []
    for i in idx:
        fr = cv2.imread(frame_files[i])
        if fr is None:
            continue
        fr = cv2.resize(fr, IMG_SIZE)
        seq.append(extract_hand_vector_from_frame(fr, hands_model))

    return pad_or_trim(seq, max_frames)

Using MediaPipe Hands from mp.solutions.hands


In [67]:
from tqdm.auto import tqdm

def build_xy(split_df, split_name):
    cache_x = os.path.join(CACHE_DIR, f"X_{split_name}.npy")
    cache_y = os.path.join(CACHE_DIR, f"y_{split_name}.npy")

    if os.path.exists(cache_x) and os.path.exists(cache_y):
        print(f"Loading cached {split_name} tensors...")
        X = np.load(cache_x)
        y = np.load(cache_y)
        return X, y

    X_list, y_list = [], []

    # Reuse one Hands model for the entire split (faster + fewer warnings)
    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as hands_model:
        for _, row in tqdm(
            split_df.iterrows(), total=len(split_df), desc=f"Extract {split_name}"
        ):
            media_path, label, source_type = (
                row["media_path"],
                row["label"],
                row["source_type"],
            )
            try:
                if source_type == "video":
                    seq = extract_sequence_from_video(
                        media_path, hands_model, max_frames=MAX_FRAMES
                    )
                else:
                    seq = extract_sequence_from_frames_dir(
                        media_path, hands_model, max_frames=MAX_FRAMES
                    )

                X_list.append(seq)
                y_list.append(label)
            except Exception as e:
                print(f"[WARN] Failed: {media_path} -> {e}")

    X = np.array(X_list, dtype=np.float32)
    y = le.transform(y_list).astype(np.int32)

    np.save(cache_x, X)
    np.save(cache_y, y)
    return X, y

X_train, y_train = build_xy(train_df, "train")
X_val, y_val = build_xy(val_df, "val")
X_test, y_test = build_xy(test_df, "test")

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

Loading cached train tensors...
Loading cached val tensors...
Loading cached test tensors...
X_train: (28, 40, 126) y_train: (28,)
X_val:   (6, 40, 126) y_val:   (6,)
X_test:  (6, 40, 126) y_test:  (6,)


In [65]:
def build_lightweight_model(timesteps=MAX_FRAMES, feat_dim=126, n_classes=5):
    inp = layers.Input(shape=(timesteps, feat_dim), name="landmarks")

    x = layers.LayerNormalization()(inp)
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(96, 3, padding="same", activation="relu")(x)
    x = layers.GRU(64, return_sequences=False)(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax")(x)

    model = tf.keras.Model(inp, out, name="asl_word_light")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_lightweight_model(
    timesteps=MAX_FRAMES, feat_dim=126, n_classes=num_classes
)
model.summary()

Model: "asl_word_light"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ landmarks (InputLayer)          │ (None, 40, 126)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_1           │ (None, 40, 126)        │           252 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 40, 64)         │        24,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 20, 96)         │        18,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        31,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 78,625 (307.13 KB)

 Trainable params: 78,625 (307.13 KB)

 Non-trainable params: 0 (0.00 B)

In [68]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=6, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/best_asl_word_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
    ),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/35
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.1429 - loss: 1.5954 - val_accuracy: 0.1667 - val_loss: 1.5737 - learning_rate: 0.0010
Epoch 2/35
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step - accuracy: 0.3214 - loss: 1.5075 - val_accuracy: 0.5000 - val_loss: 1.5260 - learning_rate: 0.0010
Epoch 3/35
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.3929 - loss: 1.4586 - val_accuracy: 0.3333 - val_loss: 1.4968 - learning_rate: 0.0010
Epoch 4/35
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.3929 - loss: 1.4111 - val_accuracy: 0.3333 - val_loss: 1.4879 - learning_rate: 0.0010
Epoch 5/35
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.4286 - loss: 1.3700 - val_accuracy: 0.3333 - val_loss: 1.4836 - learning_rate: 0.0010
Epoch 6/35
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.6429 - loss: 1.3262 - val_accuracy: 0.5000 - val_loss: 1.4844 - learning_rate: 0.0010
Epoch 7/35
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.6786 - loss: 1.2456 - val_accuracy: 0.5

In [76]:
# Balanced class weighting + minority oversampling for tiny datasets
classes = np.unique(y_train)
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
 )
class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}
print("Class weights:", class_weight_dict)

def oversample_with_landmark_jitter(X, y, noise_std=0.01):
    counts = pd.Series(y).value_counts()
    target = int(counts.max())

    X_parts = [X]
    y_parts = [y]

    for cls, cnt in counts.items():
        need = target - int(cnt)
        if need <= 0:
            continue

        idx = np.where(y == cls)[0]
        sampled_idx = np.random.choice(idx, size=need, replace=True)
        X_new = np.copy(X[sampled_idx])

        # Jitter only non-zero landmarks (keeps padded frames unchanged)
        noise = np.random.normal(0, noise_std, size=X_new.shape).astype(np.float32)
        mask = (X_new != 0).astype(np.float32)
        X_new = X_new + noise * mask

        y_new = np.full((need,), cls, dtype=y.dtype)
        X_parts.append(X_new)
        y_parts.append(y_new)

    X_bal = np.concatenate(X_parts, axis=0)
    y_bal = np.concatenate(y_parts, axis=0)

    # Shuffle
    perm = np.random.permutation(len(y_bal))
    return X_bal[perm], y_bal[perm]

X_train_bal, y_train_bal = oversample_with_landmark_jitter(X_train, y_train, noise_std=0.01)
print("Train counts (original):")
print(pd.Series(y_train).value_counts().sort_index())
print("Train counts (balanced):")
print(pd.Series(y_train_bal).value_counts().sort_index())

effective_batch_size = min(BATCH_SIZE, max(4, len(X_train_bal) // 6))
print("Effective batch size:", effective_batch_size, "| balanced train samples:", len(X_train_bal))

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=10, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/best_asl_word_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
    ),
]

history = model.fit(
    X_train_bal,
    y_train_bal,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=effective_batch_size,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,
)

Class weights: {0: 0.8, 1: 1.8666666666666667, 2: 0.7, 3: 0.8, 4: 1.8666666666666667}
Train counts (original):
0    7
1    3
2    8
3    7
4    3
Name: count, dtype: int64
Train counts (balanced):
0    8
1    8
2    8
3    8
4    8
Name: count, dtype: int64
Effective batch size: 6 | balanced train samples: 40
Epoch 1/35
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 79ms/step - accuracy: 0.4726 - loss: 2.0140 - val_accuracy: 0.5000 - val_loss: 1.4142 - learning_rate: 5.0000e-04
Epoch 2/35
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.5354 - loss: 1.7473 - val_accuracy: 0.5000 - val_loss: 1.3830 - learning_rate: 5.0000e-04
Epoch 3/35
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6858 - loss: 1.5639 - val_accuracy: 0.6667 - val_loss: 1.3600 - learning_rate: 5.0000e-04
Epoch 4/35
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9469 - loss: 1.3546 - val_accuracy: 0.6667 - val_loss: 1.2743 - learning_rate: 5.0000e-04
Epoch 5/35
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9104 - loss: 1

In [77]:
# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

y_prob = model.predict(X_test, verbose=0)
y_pred = y_prob.argmax(axis=1)

print(classification_report(y_test, y_pred, target_names=class_names))
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)

Test accuracy: 0.8333
              precision    recall  f1-score   support

       hello       1.00      1.00      1.00         1
          no       0.50      1.00      0.67         1
       sorry       1.00      1.00      1.00         1
    thankyou       1.00      1.00      1.00         2
         yes       0.00      0.00      0.00         1

    accuracy                           0.83         6
   macro avg       0.70      0.80      0.73         6
weighted avg       0.75      0.83      0.78         6

Confusion matrix:
 [[1 0 0 0 0]
 [0 1 0 0 0]
 [0 0 1 0 0]
 [0 0 0 2 0]
 [0 1 0 0 0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  if average == "binary":
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  if average == "binary":
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  if average == "binary":


In [78]:
# Save label map + export compact TFLite model for real-time inference
label_map = {int(i): c for i, c in enumerate(class_names)}
with open("/kaggle/working/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

# Save Keras model artifact
model.save("/kaggle/working/best_asl_word_model.keras")

def convert_to_tflite(keras_model, allow_select_tf_ops=False):
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    if allow_select_tf_ops:
        # Needed for some GRU/TensorList graphs
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS,
        ]
        converter._experimental_lower_tensor_list_ops = False

    return converter.convert()

tflite_path = "/kaggle/working/asl_word_light.tflite"
conversion_mode = "TFLITE_BUILTINS"

try:
    tflite_model = convert_to_tflite(model, allow_select_tf_ops=False)
except Exception as e:
    print("Built-in TFLite conversion failed. Retrying with SELECT_TF_OPS fallback...")
    print("Reason:", str(e)[:500])
    tflite_model = convert_to_tflite(model, allow_select_tf_ops=True)
    conversion_mode = "SELECT_TF_OPS_FALLBACK"

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("Saved artifacts:")
print("- /kaggle/working/best_asl_word_model.keras")
print("- /kaggle/working/asl_word_light.tflite")
print("- /kaggle/working/label_map.json")
print("TFLite conversion mode:", conversion_mode)

if conversion_mode == "SELECT_TF_OPS_FALLBACK":
    print(
        "Note: This TFLite model uses SELECT_TF_OPS (Flex). "
        "For Python webcam inference this is fine; mobile deployment may need Flex support."
    )

INFO:tensorflow:Assets written to: /tmp/tmpqbr12nyz/assets


INFO:tensorflow:Assets written to: /tmp/tmpqbr12nyz/assets


Saved artifact at '/tmp/tmpqbr12nyz'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 126), dtype=tf.float32, name='landmarks')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  132122051827408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051832592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051831824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051827600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051832400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051831248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051830288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051825872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051831632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051831056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051830480: Tens

W0000 00:00:1773893574.861541      55 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1773893574.861818      55 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
loc(callsite(callsite(fused["TensorListReserve:", "asl_word_light_1/gru_1_1/TensorArrayV2_1@__inference_function_25338"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_25401"]) at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): error: 'tf.TensorListReserve' op requires element_shape to be static during TF Lite transformation pass
loc(callsite(callsite(fused["TensorListReserve:", "asl_word_light_1/gru_1_1/TensorArrayV2_1@__inference_function_25338"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_25401"]) at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): error: failed to legalize operation 'tf.TensorListReserve' that was explicitly marked illegal
error: Lowering tens

Built-in TFLite conversion failed. Retrying with SELECT_TF_OPS fallback...
Reason: <unknown>:0: error: loc(callsite(callsite(fused["TensorListReserve:", "asl_word_light_1/gru_1_1/TensorArrayV2_1@__inference_function_25338"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_25401"]) at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): 'tf.TensorListReserve' op requires element_shape to be static during TF Lite transformation pass
<unknown>:0: note: loc(fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]): call
INFO:tensorflow:Assets written to: /tmp/tmplpvxmyc4/assets


INFO:tensorflow:Assets written to: /tmp/tmplpvxmyc4/assets


Saved artifact at '/tmp/tmplpvxmyc4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 126), dtype=tf.float32, name='landmarks')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  132122051827408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051832592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051831824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051827600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051832400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051831248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051830288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051825872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051831632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051831056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122051830480: Tens

W0000 00:00:1773893576.046412      55 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1773893576.046450      55 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


In [73]:
# Optional local webcam inference helper (run on your laptop, not Kaggle)
# This demonstrates streaming inference with the same landmark pipeline.

from collections import deque


def run_webcam_streaming_demo(keras_model, class_names, max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(0)
    buffer = deque(maxlen=max_frames)

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as hands_model:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame_small = cv2.resize(frame, IMG_SIZE)
            feat = extract_hand_vector_from_frame(frame_small, hands_model)
            buffer.append(feat)

            pred_txt = "Collecting..."
            if len(buffer) == max_frames:
                x = np.array(buffer, dtype=np.float32)[None, ...]  # [1, T, 126]
                prob = keras_model.predict(x, verbose=0)[0]
                idx = int(np.argmax(prob))
                conf = float(prob[idx])
                pred_txt = f"{class_names[idx]} ({conf:.2f})"

            cv2.putText(
                frame, pred_txt, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2
            )
            cv2.imshow("ASL Streaming Demo", frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()


# Usage on local machine:
# run_webcam_streaming_demo(model, class_names)